# ReAct agents, subagents, and trajectories (Chapter 8)

This notebook accompanies the chapter's **DSPy's ReAct Agent** section. It starts with ordinary Python functions as tools, builds signature-polymorphic agents, delegates work to a subagent, and inspects a trajectory for evaluation.

**Required environment variable**

- `OPENAI_API_KEY` — used by the default `openai/gpt-5.6-luna` LM.

Create a `.env` file in the repository root (or export the variable in your shell) before running the setup cell.


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))


## Defining tools

`dspy.ReAct` can wrap any typed Python callable. The function name, docstring, argument names, and type hints become the schema shown to the LM. The calculator uses `eval` only to mirror the compact book example; do not expose it to untrusted input in a real application.


In [ ]:
def search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"[stub] documentation results for: {query}"


def get_order_status(order_id: str) -> str:
    """Look up the current status of a customer order by its ID."""
    return f"Order {order_id}: shipped, arriving Tuesday."


def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


## A first ReAct agent

The agent runs a think/act/observe loop until it calls the built-in `finish` tool or reaches `max_iters`. DSPy then performs a final extraction step to populate the declared output fields.


In [ ]:
agent = dspy.ReAct(
    "question -> answer",
    tools=[search_knowledge_base, get_order_status, calculate],
    max_iters=10,
)

result = agent(question="What's the status of order #12345?")
print(result.answer)


## Signature-polymorphic ReAct

The signature can declare multiple typed inputs and outputs. Only the customer-service agent is run below; the research agent shows the same construction pattern for a different schema.


In [ ]:
def search_kb(query: str) -> str:
    """Search the internal knowledge base for help articles."""
    return f"[stub] kb hit for: {query}"


def get_order(order_id: str) -> str:
    """Fetch the order record for a given order ID."""
    return f"[stub] order {order_id} found."


def get_account(customer_id: str) -> str:
    """Fetch the customer account record for a given customer ID."""
    return f"[stub] account {customer_id} found."


def file_ticket(summary: str) -> str:
    """File a support ticket with a short summary."""
    return f"[stub] ticket filed: {summary}"


def web_search(query: str) -> str:
    """Search the web for current information."""
    return f"[stub] web results for: {query}"


def search_papers(query: str) -> str:
    """Search research papers for a topic."""
    return f"[stub] papers for: {query}"


service_agent = dspy.ReAct(
    "customer_query, customer_id -> response, escalation_needed: bool",
    tools=[search_kb, get_order, get_account, file_ticket],
    max_iters=8,
)

research_agent = dspy.ReAct(
    "topic -> summary, sources: list[str]",
    tools=[web_search, search_papers],
    max_iters=15,
)

service_result = service_agent(
    customer_query="My order #12345 never arrived, can you help?",
    customer_id="C-001",
)
print(service_result.response)
print("escalation_needed:", service_result.escalation_needed)


## Building a `dspy.Tool` explicitly

Most tools need no wrapper configuration. Construct `dspy.Tool` directly when you want to override its name, description, or per-argument hints.


In [ ]:
def my_function(query: str) -> str:
    """Look up documentation."""
    return f"[stub] docs lookup for: {query}"


tool = dspy.Tool(
    func=my_function,
    name="search_docs",
    desc="Search internal documentation by keyword",
    arg_desc={"query": "The search query to run against the docs"},
)

print(tool)


## Subagents as tools

A parent ReAct agent can call a function that delegates to another DSPy module. The manager records delegation as one tool call, while the worker returns its own separate trajectory.


In [ ]:
fact_checker = dspy.ReAct(
    "claim -> verdict, evidence: list[str]",
    tools=[search_knowledge_base],
    max_iters=5,
)


def verify_claim(claim: str) -> str:
    """Delegate a claim to the fact-checking agent."""
    verification = fact_checker(claim=claim)
    return (
        f"Verdict: {verification.verdict}\n"
        f"Evidence:\n" + "\n".join(verification.evidence)
    )


manager = dspy.ReAct(
    "question -> answer",
    tools=[verify_claim, calculate],
    max_iters=6,
)

delegated_result = manager(
    question="Verify that DSPy supports typed signatures and calculate 15 * 23."
)
print(delegated_result.answer)


## Tracing trajectories

Every ReAct prediction includes a flat trajectory dictionary. Each loop iteration contributes a thought, tool name, tool arguments, and observation.


In [ ]:
for i in range(20):
    if f"thought_{i}" not in result.trajectory:
        break
    print(f"Step {i + 1}:")
    print(f"  Thought: {result.trajectory[f'thought_{i}']}")
    print(f"  Tool: {result.trajectory[f'tool_name_{i}']}")
    print(f"  Args: {result.trajectory[f'tool_args_{i}']}")
    print(f"  Observation: {result.trajectory[f'observation_{i}']}")


## Evaluating efficiency

A trajectory-aware metric can reward correctness while penalizing unnecessary non-`finish` tool calls. This metric can be passed to `dspy.Evaluate` and an optimizer such as `BootstrapFewShot` or GEPA.


In [ ]:
def efficiency_metric(example, prediction, trace=None):
    """Reward correct answers and penalize unnecessary tool calls."""
    correct = prediction.answer.lower() == example.answer.lower()
    if not correct:
        return 0.0

    steps = sum(
        1
        for i in range(20)
        if f"tool_name_{i}" in prediction.trajectory
        and prediction.trajectory[f"tool_name_{i}"] != "finish"
    )

    if steps <= 2:
        return 1.0
    if steps <= 4:
        return 0.8
    return 0.5


example = dspy.Example(answer=result.answer)
print("Efficiency score:", efficiency_metric(example, result))
